In [ ]:
import pandas as pd
import numpy as np
from cca_classes import BaselineCCAPricer, compute_barrier_kvm, compute_lcl_usd
from CCA_utils import *

##Baseline Model panel:

In [4]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T=5.0
vol_window = 52
freq = 'W'

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


In [ ]:
results = pd.DataFrame()
pricer = BaselineCCAPricer()

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)
    
    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(M_bn, dom_D_bn, fx_rate, r_d, r_f)
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}

    for i, row in df.iterrows():
        cca = pricer.solve_CCA_M0(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                        r_f.iloc[i], T)
        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca['sigma_V'])
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

results[['date', 'country', 'cds_spread', 'risk_free_rate',
         'implied_V', 'implied_sigma_V', 'cca_converged',
         'B_f', 'LCL_usd', 'sigma_lcl']].to_csv(
    '../output/results/M0_results_5YCDS.csv', index=False)

In [4]:
results.to_csv("../output/results/M0_results_5YCDS.csv")